<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-01/AI_Modul_1.7_Praktikum_Etika_dan_Tata_Kelola_AI.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 1.7: Praktikum Etika, Keadilan Algoritmik, dan Tata Kelola AI (Responsible AI)
### Eksperimen Audit Keadilan Disparate Impact Ratio (DIR) & Mitigasi Bias Data Historis (Sample Reweighting)

---

> **Diktat Terkait:** [AI_Modul_1.7_Etika_dalam_Penggunaan_AI.md](../../docs/part-01/AI_Modul_1.7_Etika_dalam_Penggunaan_AI.md)  
> **Outputs:** Kode program audit keadilan matematis, perhitungan metrik Disparate Impact Ratio (DIR) dan Equalized Odds Difference (EOD), serta implementasi algoritma debiasing Sample Reweighting.  
> **Outcomes:** Mahasiswa memiliki keterampilan mendeteksi bias tersembunyi pada dataset kredit perbankan/agro-industri, melakukan audit kepatuhan regulasi (UU PDP Indonesia & EU AI Act), dan merekayasa mitigasi bias secara adil.  
> **Impacts:** Terciptanya sistem AI yang inklusif, adil, transparan, dan melindungi kelompok rentan (seperti petani swadaya) dari diskriminasi otomatisasi algoritma.
>
> **Tujuan Pembelajaran:**  
> 1. Mengidentifikasi dampak bias historis terhadap keputusan klasifikasi biner machine learning.  
> 2. Menghitung Disparate Impact Ratio (DIR) dan menguji kepatuhan terhadap kaidah hukum *Four-Fifths Rule* (aturan 80%).  
> 3. Mengimplementasikan algoritma prapemrosesan *Sample Reweighting* untuk mengembalikan keadilan statistik.  
> 4. Memvisualisasikan perbandingan tingkat persetujuan sebelum dan sesudah mitigasi bias.

## Ringkasan 7 Dimensi Responsible AI & Ragam Kasus Nyata Dunia

Sistem kecerdasan buatan modern wajib mematuhi 7 pilar etika kuantitatif dan kepatuhan hukum:
1. **Keadilan & Non-Diskriminasi (*Fairness*)**: Mencegah diskriminasi rasial/gender/status sosial (*Kasus Amazon AI Recruitment & COMPAS Recidivism Tool*).
2. **Transparansi & Keterjelasan (*Explainable AI/XAI*)**: Mencegah bahaya keputusan 'kotak hitam' (*Kasus Skandal Robodebt Australia senilai AUD $1.8 Miliar*).
3. **Privasi & Tata Kelola Data**: Melindungi data biometrik dan kedaulatan data warga (*Kasus Clearview AI 30 Miliar Wajah & Kepatuhan UU No. 27/2022 PDP Pasal 40*).
4. **Akuntabilitas & Liabilitas Hukum**: Menetapkan subjek hukum pertanggungjawaban atas kerugian fisik/materiil (*Kasus Kecelakaan Fatal Mobil Otonom Uber Tempe 2018*).
5. **Keselamatan & Dilema Penggunaan Ganda (*Dual-Use*)**: Mencegah model AI dibalik tujuannya untuk kejahatan siber atau biokimia (*Kasus AI MegaSyn 40.000 Senjata Kimia dalam 6 Jam & Voice Cloning Phishing*).
6. **Keberlanjutan Lingkungan (*Green AI*)**: Mengendalikan emisi karbon dan pemborosan air pendingin pusat data server GPU.
7. **Hak Cipta, Integritas Intelektual & Pekerja Anotasi**: Melindungi hak pencipta (*Kasus Getty Images & NYT vs AI*) serta hak upah layak bagi pelabel data (*The Ghost Workers*).

---


## 1. Persiapan Environment & Import Library
Jalankan sel berikut untuk memuat pustaka saintifik dan pemodelan statistik:

In [ ]:
# Mengimpor modul sys untuk membaca metadata sistem Python
import sys

# Mengimpor pustaka numpy untuk kalkulasi vektor dan matriks data pemohon
import numpy as np

# Mengimpor pustaka matplotlib untuk membuat grafik perbandingan keadilan
import matplotlib.pyplot as plt

# Mengimpor algoritma Logistic Regression sebagai pengambil keputusan persetujuan kredit
from sklearn.linear_model import LogisticRegression

# Mengimpor fungsi pemisahan data latih dan uji dari scikit-learn
from sklearn.model_selection import train_test_split

# Mengimpor metrik evaluasi klasifikasi: confusion matrix dan akurasi global
from sklearn.metrics import confusion_matrix, accuracy_score

# Menetapkan nilai seed acak agar hasil eksperimen selalu konsisten saat direplikasi
np.random.seed(42)

# Mengatur gaya grafik visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

# Menampilkan versi interpreter Python
print(f"Python Version : {sys.version.split()[0]}")
print("[OK] Seluruh pustaka audit etika AI berhasil dimuat dengan sukses!")

## 2. Pembangkitan Dataset Pinjaman Tani dengan Bias Historis

Kita memodelkan 1.000 pemohon pinjaman mikro peremajaan kelapa sawit:
* **Atribut Sensitif ($A$)**: $A=1$ (Petani Korporasi / *Privileged*), $A=0$ (Petani Swadaya / *Unprivileged*).
* **Fitur Objektif ($X_1, X_2$)**: Skor Kesehatan Agronomis Kebun dan Rasio Kemampuan Bayar Arus Kas.
* **Bias Historis**: Data masa lalu sengaja tercemar bias di mana petani swadaya dipersulit persetujuannya meskipun kualitas kebunnya sangat baik.

In [ ]:
# Menentukan jumlah pemohon pinjaman kredit mikro tani
n_samples = 1000

# Membangkitkan Atribut Sensitif (A): 60% Petani Korporasi (1), 40% Petani Swadaya (0)
A = np.random.binomial(n=1, p=0.60, size=n_samples)

# Membangkitkan Fitur Objektif 1: Skor Agronomis Kebun (rata-rata 70, deviasi standar 10)
skor_agronomis = np.random.normal(loc=70.0, scale=10.0, size=n_samples)

# Membangkitkan Fitur Objektif 2: Rasio Arus Kas Pembayaran (rata-rata 2.8, deviasi 0.6)
arus_kas = np.random.normal(loc=2.8, scale=0.6, size=n_samples)

# Fitur Proksi: Kelengkapan Agunan Sertifikat Formal (berkorelasi dengan korporasi)
agunan = 0.50 * A + np.random.normal(loc=1.0, scale=0.30, size=n_samples)

# Menghitung probabilitas kelayakan agronomis sejati (True Merit) tanpa bias A
prob_merit = 1.0 / (1.0 + np.exp(-(0.08 * (skor_agronomis - 70.0) + 1.2 * (arus_kas - 2.8))))

# Mencemari label masa lalu dengan bias manusia historis: korporasi diuntungkan, swadaya dipersulit
bias_faktor = np.where(A == 1, 0.15, -0.25)
prob_historis = np.clip(prob_merit + bias_faktor, 0.05, 0.95)
y_historis = (np.random.rand(n_samples) < prob_historis).astype(int)

# Menggabungkan fitur masukan menjadi matriks X (termasuk fitur proksi agunan)
X = np.column_stack([skor_agronomis, arus_kas, agunan])

# Membagi dataset menjadi 80% data latih dan 20% data uji beserta atribut sensitifnya
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X, y_historis, A, test_size=0.20, random_state=42
)

print("=" * 65)
print("DISTRIBUSI DATA PEMOHON PINJAMAN KREDIT SAWIT")
print("=" * 65)
print(f"-> Total Pemohon Data Latih: {len(y_train)} orang")
print(f"   - Petani Korporasi (A=1) : {np.sum(A_train == 1)} orang")
print(f"   - Petani Swadaya   (A=0) : {np.sum(A_train == 0)} orang")
print(f"-> Total Pemohon Data Uji  : {len(y_test)} orang")
print("=" * 65)

## 3. Audit Model Awal: Evaluasi Disparate Impact Ratio (DIR)

Kita melatih model standar tanpa mitigasi. Rumus **Disparate Impact Ratio (DIR)**:

$$\text{DIR} = \frac{P(\hat{Y} = 1 \mid A = 0)}{P(\hat{Y} = 1 \mid A = 1)}$$

Standar hukum perbankan menetapkan bahwa $\text{DIR} < 0.80$ merupakan **pelanggaran diskriminasi sistemik** (*Four-Fifths Rule Violation*).

In [ ]:
# Melatih model regresi logistik standar pada data latih yang tercemar bias historis
model_bias = LogisticRegression()
model_bias.fit(X_train, y_train)

# Melakukan prediksi kelulusan pinjaman pada data uji
y_pred_bias = model_bias.predict(X_test)

# Memisahkan keputusan prediksi untuk masing-masing kelompok data uji
mask_priv_test = (A_test == 1)
mask_unpriv_test = (A_test == 0)

rate_priv_bias = np.mean(y_pred_bias[mask_priv_test])
rate_unpriv_bias = np.mean(y_pred_bias[mask_unpriv_test])

# Menghitung rasio Disparate Impact Ratio awal
dir_awal = rate_unpriv_bias / rate_priv_bias

print("=" * 65)
print("HASIL AUDIT KEADILAN MODEL AWAL (BASELINE MODEL)")
print("=" * 65)
print(f"-> Approval Rate Korporasi (Privileged)   : {rate_priv_bias * 100:.2f}%")
print(f"-> Approval Rate Swadaya   (Unprivileged) : {rate_unpriv_bias * 100:.2f}%")
print(f"-> Disparate Impact Ratio (DIR)           : {dir_awal:.4f}")
print("-" * 65)
if dir_awal < 0.80:
    print(">> [VONIS HUKUM]: MODEL DISKRIMINATIF ILEGAL! (DIR < 0.80)")
    print("   Model melanggar aturan Four-Fifths Rule dan Pasal 40 UU PDP!")
else:
    print(">> [VONIS HUKUM]: Model Lolos Ambang Batas Keadilan Hukum.")
print("=" * 65)

## 4. Algoritma Mitigasi Bias: Pre-Processing Sample Reweighting

Untuk memulihkan keadilan tanpa menghilangkan variabel objektif, kita menerapkan metode pembobotan sampel (Kamiran & Calders, 2012):

$$W(A=a, Y=y) = \frac{N_a \times N_y}{N \times N_{a, y}}$$

In [ ]:
# Menghitung total sampel latih
N = len(y_train)
bobot_sampel = np.zeros(N)

print("=" * 65)
print("KALKULASI BOBOT SAMPEL KEADILAN (KAMIRAN & CALDERS)")
print("=" * 65)

# Mengalkulasi bobot reweighting untuk setiap kombinasi kelompok dan target
for a_val in [0, 1]:
    for y_val in [0, 1]:
        mask = (A_train == a_val) & (y_train == y_val)
        n_ay = np.sum(mask)
        n_a = np.sum(A_train == a_val)
        n_y = np.sum(y_train == y_val)
        
        bobot = (n_a * n_y) / (N * n_ay)
        bobot_sampel[mask] = bobot
        
        label_kelompok = "Korporasi" if a_val == 1 else "Swadaya"
        label_status = "Disetujui" if y_val == 1 else "Ditolak"
        print(f"-> Bobot [{label_kelompok:<9} | {label_status:<9}]: {bobot:.4f} (Populasi = {n_ay})")
print("=" * 65)

## 5. Melatih Model Adil (Fair Model) & Pembuktian Pemulihan Keadilan

Kita melatih kembali model regresi logistik menggunakan parameter `sample_weight=bobot_sampel`.

In [ ]:
# Menginisialisasi dan melatih model adil dengan pembobotan sampel
model_fair = LogisticRegression()
model_fair.fit(X_train, y_train, sample_weight=bobot_sampel)

# Melakukan prediksi kelulusan pinjaman dengan model adil
y_pred_fair = model_fair.predict(X_test)

# Menghitung ulang tingkat persetujuan pasca-mitigasi pada data uji
rate_priv_fair = np.mean(y_pred_fair[mask_priv_test])
rate_unpriv_fair = np.mean(y_pred_fair[mask_unpriv_test])

# Menghitung nilai DIR baru pasca-mitigasi
dir_fair = rate_unpriv_fair / rate_priv_fair

# Memeriksa perbandingan akurasi global
acc_awal = accuracy_score(y_test, y_pred_bias)
acc_fair = accuracy_score(y_test, y_pred_fair)

print("=" * 65)
print("HASIL AUDIT KEADILAN PASCA-MITIGASI (FAIR MODEL)")
print("=" * 65)
print(f"-> Approval Rate Baru Korporasi  : {rate_priv_fair * 100:.2f}%")
print(f"-> Approval Rate Baru Swadaya    : {rate_unpriv_fair * 100:.2f}%")
print(f"-> Disparate Impact Ratio (DIR)  : {dir_fair:.4f}")
print(f"-> Akurasi Bisnis Global         : {acc_fair * 100:.2f}% (Awal: {acc_awal * 100:.2f}%)")
print("-" * 65)
if dir_fair >= 0.80:
    print(">> [VONIS HUKUM]: REKAYASA BERHASIL! Model Lolos Audit Keadilan (DIR >= 0.80)")
    print("   Model mematuhi regulasi UU PDP dan standar etika Responsible AI.")
else:
    print(">> [VONIS HUKUM]: Model Belum Memenuhi Batas Keadilan.")
print("=" * 65)

## 6. Visualisasi Komparasi Keadilan Sebelum vs Sesudah Mitigasi

In [ ]:
# Visualisasi grafik perbandingan tingkat persetujuan dan rasio DIR
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.8))

# Subplot 1: Bar Chart Approval Rate per Kelompok
kategori = ['Petani Korporasi', 'Petani Swadaya']
x = np.arange(len(kategori))
lebar = 0.35

bar1 = ax1.bar(x - lebar/2, [rate_priv_bias*100, rate_unpriv_bias*100], lebar, label='Sebelum Mitigasi (Bias)', color='#d62728', alpha=0.85)
bar2 = ax1.bar(x + lebar/2, [rate_priv_fair*100, rate_unpriv_fair*100], lebar, label='Sesudah Mitigasi (Fair)', color='#2ca02c', alpha=0.85)

ax1.set_ylabel('Tingkat Persetujuan Pinjaman (%)', fontsize=10, fontweight='bold')
ax1.set_title('Tingkat Persetujuan Kredit per Kelompok', fontsize=11, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(kategori, fontsize=10, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.legend(loc='lower left')

# Subplot 2: Bar Chart Disparate Impact Ratio (DIR)
bars_dir = ax2.bar(['Sebelum Mitigasi', 'Sesudah Mitigasi'], [dir_awal, dir_fair], 
                   color=['#d62728', '#2ca02c'], width=0.45, alpha=0.85, edgecolor='black', linewidth=1.2)

ax2.axhline(0.80, color='blue', linestyle='--', linewidth=2, label='Batas Minimum Legal (Four-Fifths Rule = 0.80)')
ax2.set_ylabel('Disparate Impact Ratio (DIR)', fontsize=10, fontweight='bold')
ax2.set_title('Evaluasi Kepatuhan Hukum Keadilan (DIR)', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.15)
ax2.legend(loc='upper left')

for b in bars_dir:
    yval = b.get_height()
    ax2.text(b.get_x() + b.get_width()/2.0, yval + 0.03, f"{yval:.3f}", ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle("Audit Keadilan Algoritmik: Dampak Mitigasi Sample Reweighting", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Tugas Mandiri & Eksplorasi Mahasiswa

1. **Audit Equal Opportunity**: Hitung nilai *True Positive Rate* (TPR) dari petani swadaya vs korporasi pada model awal dan model adil. Berapakah selisih nilainya?
2. **Eksplorasi Fitur Proksi**: Jika fitur `A` (Kelompok Tani) dihapus total dari proses pelatihan, apakah model otomatis menjadi adil? Ujilah hipotesis ini (*Fairness through Unawareness*) dengan melatih model tanpa variabel `A` dan periksa kembali skor DIR-nya!